In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.utils.prune as prune  
import os


class ConvNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, stride=2)

        self.fc1 = nn.Linear(128 * 10 * 10, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def apply_structured_pruning(model, amount=0.3):
   
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if name == 'fc2': 
                continue
            prune.ln_structured(module, name="weight", amount=amount, n=1, dim=0)
            prune.remove(module, 'weight')
    print(f"--- Structured Pruning Applied: {amount*100}% of filters/neurons removed ---")


class DualTransform:
    def __init__(self, weak, strong):
        self.weak = weak
        self.strong = strong
    def __call__(self, x):
        return self.weak(x), self.strong(x)

def get_loaders(data_path, batch_size):
    l_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(96, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    weak_t = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    strong_t = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    train_set = torchvision.datasets.STL10(root=data_path, split='train', download=True, transform=l_transform)
    unlabeled_set = torchvision.datasets.STL10(root=data_path, split='unlabeled', download=True, transform=DualTransform(weak_t, strong_t))
    test_set = torchvision.datasets.STL10(root=data_path, split='test', download=True, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    un_loader = torch.utils.data.DataLoader(unlabeled_set, batch_size=batch_size*4, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, un_loader, test_loader


def train_and_evaluate():
    DATA_PATH = './data'
    EPOCHS = 40
    BATCH_SIZE = 64
    THRESHOLD = 0.95
    SAVE_PATH = "fixmatch_pruned_model.pth"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, un_loader, test_loader = get_loaders(DATA_PATH, BATCH_SIZE)
    
    net = ConvNeuralNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    for epoch in range(EPOCHS):
        net.train()
        
        u_iter = iter(un_loader)
        running_sup_loss = 0.0
        unsup_count = 0

        for i, (l_inputs, l_labels) in enumerate(train_loader):
            l_inputs, l_labels = l_inputs.to(device), l_labels.to(device)
            
            
            try:
                (u_weak, u_strong), _ = next(u_iter)
            except StopIteration:
                u_iter = iter(un_loader)
                (u_weak, u_strong), _ = next(u_iter)
            
            u_weak, u_strong = u_weak.to(device), u_strong.to(device)

            optimizer.zero_grad()

           
            outputs = net(l_inputs)
            loss = criterion(outputs, l_labels)

            
            with torch.no_grad():
                u_outputs_weak = net(u_weak)
                probs = torch.softmax(u_outputs_weak, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                mask = max_probs > THRESHOLD

            u_loss = torch.tensor(0.0).to(device)
            if mask.any():
                outputs_strong = net(u_strong[mask])
                u_loss = criterion(outputs_strong, pseudo_labels[mask])
                unsup_count += 1
            
            total_batch_loss = loss + (0.5 * u_loss)
            total_batch_loss.backward()
            optimizer.step()

            running_sup_loss += loss.item()


        net.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = net(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_sup_loss/len(train_loader):.4f} | Acc: {100*correct/total:.2f}%")

  
    apply_structured_pruning(net, amount=0.3)

    torch.save(net.state_dict(), SAVE_PATH)
    print(f"Pruned model saved to {SAVE_PATH}")
    
def get_model_size_mb(model):
   
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    
   
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    
   
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb

net=ConvNeuralNet()
model_size = get_model_size_mb(net)
print(f'Model size: {model_size:.3f} MB')

if __name__ == '__main__':
    train_and_evaluate()

Model size: 12.870 MB


100%|██████████| 2.64G/2.64G [01:05<00:00, 40.5MB/s]


Epoch [1/40] | Loss: 2.3988 | Acc: 29.36%
Epoch [2/40] | Loss: 1.8995 | Acc: 35.33%
Epoch [3/40] | Loss: 1.8551 | Acc: 35.45%
Epoch [4/40] | Loss: 1.8383 | Acc: 31.71%
Epoch [5/40] | Loss: 1.7824 | Acc: 33.46%
Epoch [6/40] | Loss: 1.7788 | Acc: 37.89%
Epoch [7/40] | Loss: 1.6814 | Acc: 38.67%
Epoch [8/40] | Loss: 1.6965 | Acc: 39.16%
Epoch [9/40] | Loss: 1.7078 | Acc: 37.86%
Epoch [10/40] | Loss: 1.7023 | Acc: 38.71%
Epoch [11/40] | Loss: 1.7164 | Acc: 42.85%
Epoch [12/40] | Loss: 1.6548 | Acc: 41.38%
Epoch [13/40] | Loss: 1.6315 | Acc: 44.98%
Epoch [14/40] | Loss: 1.6165 | Acc: 42.61%
Epoch [15/40] | Loss: 1.5919 | Acc: 46.16%
Epoch [16/40] | Loss: 1.5872 | Acc: 46.64%
Epoch [17/40] | Loss: 1.5224 | Acc: 47.69%
Epoch [18/40] | Loss: 1.5522 | Acc: 46.92%
Epoch [19/40] | Loss: 1.5214 | Acc: 47.46%
Epoch [20/40] | Loss: 1.5399 | Acc: 48.44%
Epoch [21/40] | Loss: 1.4987 | Acc: 50.49%
Epoch [22/40] | Loss: 1.4827 | Acc: 52.70%
Epoch [23/40] | Loss: 1.4987 | Acc: 49.33%
Epoch [24/40] | Loss